# Omni Stick — Early Stumble & Fall Detection

Predicts a fall **before** impact from IMU (motion sensor) data, using the KFall dataset.

**Pipeline:** load KFall CSVs → slice into windows → label windows shortly *before* fall onset as "at risk" → train a small 1D-CNN → evaluate → shrink to `.tflite` for the stick's microcontroller → plot the "it saw it coming" timeline.

> Run cells top to bottom. Steps that need KFall data are marked ⚠️ — everything else runs immediately.

## 1. Setup

In [ ]:
!pip install numpy pandas scikit-learn tensorflow matplotlib -q

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import glob, os

print("TensorFlow:", tf.__version__)

## 2. Get the data in ⚠️

Using the [KFall dataset mirror on Kaggle](https://www.kaggle.com/datasets/usmanabbasi2002/kfall-dataset) — no access-request wait, downloads straight into Colab.

**One-time setup (do this once, outside Colab):**
1. Create a free account at [kaggle.com](https://kaggle.com) if you don't have one.
2. Go to **Account settings → API → Create New Token**. This downloads a `kaggle.json` file to your computer.

**Every time you open this notebook fresh:** run the two cells below — the first asks you to upload `kaggle.json`, the second downloads and unzips the dataset.

In [ ]:
from google.colab import files

print("Upload your kaggle.json file (from Kaggle → Account → API → Create New Token):")
uploaded = files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!pip install kaggle -q

DATA_DIR = "/content/kfall"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d usmanabbasi2002/kfall-dataset -p $DATA_DIR --unzip

print("Files found:", len(glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)))

**If you'd rather use Google Drive instead** (e.g. the Kaggle mirror is missing files, or you've downloaded the full official dataset from the [KFall site](https://sites.google.com/view/kfalldataset)), swap the cell above for this:

```python
from google.colab import drive
drive.mount('/content/drive')

KFALL_ZIP_PATH = "/content/drive/MyDrive/kfall_data.zip"  # change to your file
DATA_DIR = "/content/kfall"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !unzip -q "$KFALL_ZIP_PATH" -d $DATA_DIR

print("Files found:", len(glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)))
```

## 3. Load one trial (sanity check) ⚠️

Before processing everything, look at a single file to confirm the column names match what we expect. **KFall's real column names may differ slightly — adjust `SENSOR_COLS` once you see the real file.**

In [ ]:
sample_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)
df_sample = pd.read_csv(sample_files[0])
print(df_sample.columns.tolist())
df_sample.head()

In [ ]:
# Adjust these once you've seen the real column names above
SENSOR_COLS = ['AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ']

WINDOW_SIZE = 100   # ~1 second at 100Hz — adjust to KFall's real sampling rate
STEP = 20            # slide the window forward by 20 readings each time
LEAD_TIME = 40        # label a window "at risk" if it ends within this many frames before fall onset

## 4. Windowing

Chops a long recording into overlapping short clips — the model looks at short snapshots of movement, not the whole recording at once.

In [ ]:
def make_windows(signal, window_size, step):
    """Slice a (T, C) signal array into overlapping windows of shape (window_size, C)."""
    windows = []
    end_indices = []
    for start in range(0, len(signal) - window_size, step):
        windows.append(signal[start:start + window_size])
        end_indices.append(start + window_size)
    return np.array(windows), np.array(end_indices)

## 5. Labeling — the key trick

A window is labeled **1 ("fall coming")** if it ends shortly *before* the labeled fall onset — not during or after. This is what turns the model from a fall *detector* into a fall *predictor*.

This function assumes a label file/column gives you the fall onset frame index per trial. **Adjust to match KFall's actual label format** (KFall ships a separate label spreadsheet per subject with onset/impact frame numbers — check the dataset's README).

In [ ]:
def label_windows(end_indices, fall_onset_idx, lead_time=LEAD_TIME):
    """1 if window ends within lead_time frames before fall onset, else 0.
    fall_onset_idx=None means this trial has no fall (a normal ADL trial) -> all zeros.
    """
    if fall_onset_idx is None:
        return np.zeros(len(end_indices), dtype=int)
    labels = np.zeros(len(end_indices), dtype=int)
    at_risk = (end_indices >= fall_onset_idx - lead_time) & (end_indices < fall_onset_idx)
    labels[at_risk] = 1
    return labels

## 6. Build the full dataset from all trials ⚠️

This loop needs to match KFall's real folder/label structure — fill in `load_trial()` once you've inspected the dataset layout in Step 3.

In [ ]:
def load_trial(csv_path):
    """Returns (signal array of shape (T, C), fall_onset_idx or None, subject_id).
    TODO: fill in real parsing once KFall's file/label structure is confirmed.
    """
    df = pd.read_csv(csv_path)
    signal = df[SENSOR_COLS].values
    fall_onset_idx = None  # TODO: look up from the matching label file
    subject_id = os.path.basename(csv_path).split('_')[0]  # TODO: adjust to real filename pattern
    return signal, fall_onset_idx, subject_id

all_X, all_y, all_subjects = [], [], []

for csv_path in sample_files:
    signal, fall_onset_idx, subject_id = load_trial(csv_path)
    windows, end_indices = make_windows(signal, WINDOW_SIZE, STEP)
    if len(windows) == 0:
        continue
    labels = label_windows(end_indices, fall_onset_idx)
    all_X.append(windows)
    all_y.append(labels)
    all_subjects.extend([subject_id] * len(windows))

X = np.concatenate(all_X)
y = np.concatenate(all_y)
subjects = np.array(all_subjects)

print("X shape:", X.shape, "| y shape:", y.shape, "| positives:", y.sum())

## 7. Train/test split — by subject, not randomly

Splitting by *person* (not randomly across all windows) tests the model on people it has never seen — a random split would make it look better than it really is.

In [ ]:
unique_subjects = np.unique(subjects)
train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)

train_mask = np.isin(subjects, train_subjects)
test_mask = np.isin(subjects, test_subjects)

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {X_train.shape[0]} windows from {len(train_subjects)} people")
print(f"Test:  {X_test.shape[0]} windows from {len(test_subjects)} people")

## 8. The model — a small 1D-CNN

Kept small on purpose so it can later be shrunk to run on a microcontroller in the stick's handle.

In [ ]:
num_channels = X_train.shape[2]

model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(16, 5, activation='relu', input_shape=(WINDOW_SIZE, num_channels)),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(32, 5, activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 9. Train

`class_weight` upweights the rare "fall coming" class so the model doesn't just lazily predict "normal" every time.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight={0: 1, 1: 5}
)

## 10. Evaluate

Accuracy alone is misleading since falls are rare — look at precision/recall for class 1 ("fall coming").

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=['normal', 'fall coming']))

## 11. "Report card" — non-technical summary

Translates the confusion matrix into the plain-language framing for your deck (caught early / caught late / missed).

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

total_falls = tp + fn
print(f"Out of {total_falls} 'fall coming' windows in the test set:")
print(f"  🟢 Caught: {tp} ({100*tp/total_falls:.0f}%)")
print(f"  🔴 Missed: {fn} ({100*fn/total_falls:.0f}%)")
print(f"\nFalse alarms on normal movement: {fp} out of {tn+fp}")

## 12. The demo visual — "it saw it coming" timeline

Runs the model second-by-second across one real fall trial and plots the rising risk score, with the fall onset, impact, and the model's own lock-trigger point marked. This is the key visual for your presentation.

In [ ]:
def plot_risk_timeline(signal, fall_onset_idx, impact_idx, model, window_size=WINDOW_SIZE, step=5, threshold=0.5):
    windows, end_indices = make_windows(signal, window_size, step)
    risk_scores = model.predict(windows).flatten()

    plt.figure(figsize=(10, 4))
    plt.plot(end_indices, risk_scores, label='Risk score', color='steelblue')
    plt.axhline(threshold, color='gray', linestyle=':', label='Lock threshold')

    if fall_onset_idx is not None:
        plt.axvline(fall_onset_idx, color='orange', label='Fall begins')
    if impact_idx is not None:
        plt.axvline(impact_idx, color='red', label='Impact')

    above = np.where(risk_scores > threshold)[0]
    if len(above) > 0:
        lock_idx = end_indices[above[0]]
        plt.axvline(lock_idx, color='green', linestyle='--', label='Stick locks')

    plt.xlabel('Time (frames)')
    plt.ylabel('Predicted fall risk')
    plt.title('Omni Stick: predicted fall risk over time')
    plt.legend()
    plt.tight_layout()
    plt.savefig('/content/risk_timeline.png', dpi=150)
    plt.show()

# Example call once you have a real trial + its onset/impact frame indices:
# plot_risk_timeline(signal, fall_onset_idx=320, impact_idx=360, model=model)

## 13. Shrink for the microcontroller

Compresses the trained model into a `.tflite` file small and fast enough to run on the chip in the stick's handle.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('/content/fall_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Saved fall_model.tflite ({len(tflite_model)/1024:.1f} KB)")

## Next steps
- Fill in `SENSOR_COLS`, `load_trial()`, and the fall-onset label lookup once KFall is unzipped and you've inspected the real file/label format (Step 3).
- Confirm the real sampling rate and adjust `WINDOW_SIZE` / `LEAD_TIME` to match it.
- Run Step 12 on a real fall trial for your presentation visual.